# 🔥 Burnout Risk Prediction System
## Pipeline Data Scientist — Capstone Project Coding Camp 2026
**Tim:** CC26-PSU335 | **Tema:** Healthy Lives & Well-being

---

### 📋 Daftar Isi
1. [Setup & Import Library](#bagian-1)
2. [Problem Discovery & Analisis Permasalahan](#bagian-2)
3. [Data Wrangling](#bagian-3)
4. [Business Questions](#bagian-4)
5. [Exploratory Data Analysis (EDA)](#bagian-5)
6. [Visualisasi & Explanatory Analysis](#bagian-6)
7. [Feature Engineering & Data Dictionary](#bagian-7)
8. [Persiapan Data untuk Model](#bagian-8)
9. [A/B Testing](#bagian-9)
10. [Simpan Artifacts](#bagian-10)

---
> **Petunjuk:** Jalankan setiap cell secara berurutan dari atas ke bawah.  
> Hasil output dan visualisasi akan muncul langsung di bawah setiap cell.


---
<a id="bagian-1"></a>
## Bagian 1 — Setup & Import Library

Pada cell ini kita mengimpor semua library yang dibutuhkan untuk seluruh pipeline analisis.

| Library | Fungsi |
|---------|--------|
| `pandas` | Manipulasi & analisis data tabular |
| `numpy` | Komputasi numerik |
| `matplotlib` / `seaborn` | Visualisasi data |
| `sklearn` | Machine Learning: preprocessing, modeling, evaluasi |
| `scipy.stats` | Uji statistik (t-test untuk A/B Testing) |
| `pickle` | Menyimpan & memuat model |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import pickle
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    accuracy_score, f1_score, precision_score, recall_score
)

# ── Konfigurasi Path ──────────────────────────────────────────────────────────
DATA_PATH  = "synthetic_employee_burnout.csv"   # sesuaikan jika path berbeda
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Tema Visualisasi ──────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="husl", font_scale=1.1)
PALETTE    = ["#2196F3", "#F44336", "#4CAF50", "#FF9800", "#9C27B0", "#00BCD4"]
COLOR_NO   = "#2196F3"   # Tidak Burnout → biru
COLOR_YES  = "#F44336"   # Burnout → merah

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "white"

print("✅ Semua library berhasil diimport!")
print(f"   pandas  v{pd.__version__}")
print(f"   numpy   v{np.__version__}")
print(f"   sklearn v{__import__('sklearn').__version__}")


---
<a id="bagian-2"></a>
## Bagian 2 — Problem Discovery & Analisis Permasalahan

### Konteks Bisnis
**Burnout karyawan** adalah kondisi kelelahan fisik dan mental akibat stres kerja berkepanjangan. 
Masalah ini seringkali tidak terdeteksi secara dini, sehingga menyebabkan:

- 📉 Produktivitas turun **30–40%**
- 🚪 Turnover karyawan meningkat (biaya rekrut = 50–200% gaji tahunan)
- 🧠 Gangguan kesehatan mental jangka panjang
- 💸 Kerugian finansial perusahaan

### Permasalahan yang Diidentifikasi (3 Kandidat)
| No | Masalah | Prioritas |
|----|---------|-----------|
| P1 | Beban kerja berlebih tanpa monitoring memadai | Tinggi |
| P2 | Tidak ada mekanisme deteksi dini burnout berbasis data | **Dipilih** |
| P3 | Intervensi HR bersifat reaktif, bukan preventif | Sedang |

### ✅ Solusi yang Dipilih: **P2**
Membangun **Burnout Risk Prediction System** berbasis Machine Learning yang mengklasifikasikan 
risiko burnout karyawan menjadi **Rendah / Sedang / Tinggi**.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# BAGIAN 2: PROBLEM DISCOVERY & ANALISIS PERMASALAHAN
# ═══════════════════════════════════════════════════════════════

problem_statement = {
    "Masalah Utama": (
        "Burnout karyawan tidak terdeteksi secara dini, menyebabkan "
        "penurunan produktivitas dan tingginya turnover."
    ),
    "Akar Penyebab": [
        "Beban kerja berlebih (WorkHoursPerWeek tinggi)",
        "Tingkat stres tidak terkelola (StressLevel tinggi)",
        "Kepuasan kerja rendah (SatisfactionLevel rendah)",
        "Ketidakseimbangan work-life (RemoteRatio tidak optimal)",
        "Faktor demografis & pengalaman kerja",
    ],
    "Solusi Dipilih": (
        "Burnout Risk Prediction System — model ML untuk klasifikasi "
        "risiko (Rendah/Sedang/Tinggi) + dashboard interaktif Streamlit."
    ),
    "Stakeholder": ["HR Manager", "Manajer Tim", "Karyawan Individu"],
    "Success Metrics": "Accuracy ≥ 85%, AUC-ROC ≥ 0.85, F1 ≥ 0.70",
}

print("=" * 60)
print("  PROBLEM STATEMENT — BURNOUT RISK PREDICTION SYSTEM")
print("=" * 60)

for k, v in problem_statement.items():
    if isinstance(v, list):
        print(f"\n  ▸ {k}:")
        for item in v:
            print(f"      - {item}")
    else:
        print(f"\n  ▸ {k}:\n      {v}")

print("\n" + "=" * 60)
print("  ✅ Problem Discovery selesai")
print("=" * 60)


---
<a id="bagian-3"></a>
## Bagian 3 — Data Wrangling

Data Wrangling mencakup 3 sub-proses:

### 3a. Gathering Data
Mengumpulkan dataset yang relevan dari sumber yang tersedia.

### 3b. Assessing Data
Mengevaluasi kualitas data: missing values, duplikat, distribusi, outlier.

### 3c. Cleaning Data
Membersihkan dan mempersiapkan data untuk analisis & modeling.


In [ ]:
# ── 3a. GATHERING DATA ───────────────────────────────────────────────────────
print("=" * 60)
print("  3a. GATHERING DATA")
print("=" * 60)

print("""
  Sumber Data:
  Dataset 'Synthetic Employee Burnout' — dataset sintetis berbasis
  distribusi statistik nyata dari literatur kesehatan mental kerja.
  Sumber: Kaggle | Format: CSV | Lisensi: Open untuk penelitian

  Relevansi:
  ✓ Fitur relevan: usia, jam kerja, stres, kepuasan
  ✓ Target jelas: Burnout (0/1)
  ✓ Ukuran memadai: 2.000 baris
""")

df = pd.read_csv(DATA_PATH)

print(f"  ✓ Dataset dimuat: {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"\n  Kolom & Tipe Data:")
print(df.dtypes.to_frame(name='Tipe').to_string())
print(f"\n  5 Baris Pertama:")
df.head()


In [ ]:
# ── 3b. ASSESSING DATA ───────────────────────────────────────────────────────
print("=" * 60)
print("  3b. ASSESSING DATA")
print("=" * 60)

# CHECK 1: Missing Values
print("\n  [CHECK 1] Missing Values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("  → ✅ Tidak ada missing values!")
else:
    print(missing[missing > 0])

# CHECK 2: Duplikasi
dupl = df.duplicated().sum()
print(f"\n  [CHECK 2] Baris Duplikat: {dupl}")
print(f"  → {'✅ Tidak ada duplikat' if dupl == 0 else f'⚠️ {dupl} baris perlu dihapus'}")

# CHECK 3: Distribusi Target
print("\n  [CHECK 3] Distribusi Target (Burnout):")
vc = df["Burnout"].value_counts()
for val, cnt in vc.items():
    label = "Burnout    " if val == 1 else "Tidak Burnout"
    print(f"    {label} (={val}): {cnt:,} ({cnt/len(df)*100:.1f}%)")

print(f"\n  ⚠️  Dataset IMBALANCED: hanya {vc.get(1,0)/len(df)*100:.1f}% yang burnout.")
print("      Perlu: stratified split + class_weight saat modeling")

# CHECK 4: Nilai Unik Kategorikal
print("\n  [CHECK 4] Nilai Unik Kolom Kategorikal:")
for col in ["Gender", "JobRole"]:
    print(f"    {col}: {df[col].unique().tolist()}")

# Statistik Deskriptif
print("\n  [CHECK 5] Statistik Deskriptif:")
df.describe().round(2)


In [ ]:
# ── 3c. CLEANING DATA ────────────────────────────────────────────────────────
print("=" * 60)
print("  3c. CLEANING DATA")
print("=" * 60)

df_clean = df.copy()

# Langkah 1: Hapus Duplikat
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"  → Duplikat dihapus: {before - len(df_clean)} baris")

# Langkah 2: Hapus Kolom Name
df_clean = df_clean.drop(columns=["Name"])
print("  → Kolom 'Name' dihapus (hanya identifier, tidak informatif)")

# Langkah 3: Imputasi (preventif)
num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
cat_cols = df_clean.select_dtypes(include="object").columns.tolist()
imputed  = 0
for col in num_cols:
    n = df_clean[col].isnull().sum()
    if n > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        print(f"  → Imputasi numerik '{col}' → median ({n} nilai)")
        imputed += 1
for col in cat_cols:
    n = df_clean[col].isnull().sum()
    if n > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
        print(f"  → Imputasi kategorikal '{col}' → modus ({n} nilai)")
        imputed += 1
if imputed == 0:
    print("  → ✅ Tidak ada missing values yang perlu diimputasi")

# Langkah 4: Validasi Range Nilai
print("\n  [VALIDASI RANGE NILAI]")
rules = {
    "Age": (22, 60), "Experience": (0, 40), "WorkHoursPerWeek": (30, 80),
    "RemoteRatio": (0, 100), "SatisfactionLevel": (1.0, 5.0),
    "StressLevel": (1, 10), "Burnout": (0, 1),
}
for col, (lo, hi) in rules.items():
    n_inv = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    status = "✅" if n_inv == 0 else f"⚠️ {n_inv} nilai di luar range"
    print(f"    {col:<22} [{lo:>4}, {hi:>4}] : {status}")

# Langkah 5: Buat kolom RiskLevel
def risk_label(row):
    """
    Scoring multi-faktor untuk klasifikasi tingkat risiko burnout.
    Score ≥ 4 → Tinggi | Score 2-3 → Sedang | Score 0-1 → Rendah
    
    Faktor:
    - WorkHoursPerWeek: ≥55 (+2), ≥45 (+1)
    - StressLevel:      ≥7  (+2), ≥4  (+1)
    - SatisfactionLevel: ≤2.0 (+2), ≤3.5 (+1)
    """
    score = 0
    if row["WorkHoursPerWeek"] >= 55: score += 2
    elif row["WorkHoursPerWeek"] >= 45: score += 1
    if row["StressLevel"] >= 7: score += 2
    elif row["StressLevel"] >= 4: score += 1
    if row["SatisfactionLevel"] <= 2.0: score += 2
    elif row["SatisfactionLevel"] <= 3.5: score += 1
    return "Tinggi" if score >= 4 else ("Sedang" if score >= 2 else "Rendah")

df_clean["RiskLevel"] = df_clean.apply(risk_label, axis=1)

print("\n  → Kolom 'RiskLevel' berhasil dibuat:")
risk_dist = df_clean["RiskLevel"].value_counts()
for lvl in ["Rendah", "Sedang", "Tinggi"]:
    cnt = risk_dist.get(lvl, 0)
    print(f"    {lvl:<8}: {cnt:,} ({cnt/len(df_clean)*100:.1f}%)")

print(f"\n  ✅ Dataset bersih: {df_clean.shape[0]:,} baris × {df_clean.shape[1]} kolom")
df_clean.head()


---
<a id="bagian-4"></a>
## Bagian 4 — Business Questions

7 pertanyaan bisnis dirancang berjenjang dari **umum → menengah → spesifik** untuk memastikan analisis terarah dan menjawab kebutuhan stakeholder.

| Kode | Level | Pertanyaan |
|------|-------|------------|
| BQ1 | Umum | Seberapa besar proporsi karyawan yang burnout? |
| BQ2 | Umum | Bagaimana distribusi burnout antar JobRole? |
| BQ3 | Umum | Adakah perbedaan burnout antar gender? |
| BQ4 | Menengah | Apakah beban kerja berkorelasi dengan burnout? |
| BQ5 | Menengah | Bagaimana kombinasi stres tinggi + jam kerja panjang mempengaruhi risiko? |
| BQ6 | Spesifik | Faktor apa yang paling berpengaruh terhadap burnout? |
| BQ7 | Spesifik | Apakah kepuasan kerja rendah berkorelasi dengan stres & burnout? |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# BAGIAN 4: BUSINESS QUESTIONS — JAWABAN RINGKAS BERBASIS DATA
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("  BUSINESS QUESTIONS — JAWABAN BERBASIS DATA")
print("=" * 60)

# BQ1
br_overall = df_clean["Burnout"].mean() * 100
print(f"\n  [BQ1] Burnout rate keseluruhan: {br_overall:.2f}%")
print(f"        {int(df_clean['Burnout'].sum())} dari {len(df_clean):,} karyawan mengalami burnout")

# BQ2
br_role = (df_clean.groupby("JobRole")["Burnout"].mean() * 100).sort_values(ascending=False)
print(f"\n  [BQ2] Burnout rate per JobRole:")
for role, rate in br_role.items():
    bar = "█" * int(rate * 2)
    print(f"    {role:<12}: {rate:>5.1f}%  {bar}")

# BQ3
br_gender = df_clean.groupby("Gender")["Burnout"].mean() * 100
print(f"\n  [BQ3] Burnout rate per Gender:")
for g, rate in br_gender.items():
    print(f"    {g:<8}: {rate:.1f}%")
print(f"    Perbedaan: {abs(br_gender.max()-br_gender.min()):.2f}%")

# BQ4
corr_w = df_clean["WorkHoursPerWeek"].corr(df_clean["Burnout"])
print(f"\n  [BQ4] Korelasi WorkHoursPerWeek ↔ Burnout: {corr_w:.4f}")
print(f"    → Korelasi {'positif' if corr_w > 0 else 'negatif'}, "
      f"{'lemah' if abs(corr_w) < 0.3 else 'sedang' if abs(corr_w) < 0.6 else 'kuat'}")

# BQ5
high_risk_br = df_clean[
    (df_clean["StressLevel"] >= 7) & (df_clean["WorkHoursPerWeek"] >= 50)
]["Burnout"].mean() * 100
low_risk_br = df_clean[
    (df_clean["StressLevel"] < 7) & (df_clean["WorkHoursPerWeek"] < 45)
]["Burnout"].mean() * 100
print(f"\n  [BQ5] Burnout rate (Stres≥7 & Jam≥50): {high_risk_br:.1f}%")
print(f"        Burnout rate (Stres<7 & Jam<45):  {low_risk_br:.1f}%")
print(f"        Perbedaan: {high_risk_br - low_risk_br:+.1f} poin persentase  🚨")

# BQ7
corr_sat   = df_clean["SatisfactionLevel"].corr(df_clean["Burnout"])
corr_stress = df_clean["StressLevel"].corr(df_clean["Burnout"])
print(f"\n  [BQ7] Korelasi SatisfactionLevel ↔ Burnout : {corr_sat:.4f}  (negatif = makin puas → makin jarang burnout)")
print(f"        Korelasi StressLevel ↔ Burnout         : {corr_stress:.4f}  (positif = makin stres → makin sering burnout)")

print("\n  ✅ Preview Business Questions selesai — visualisasi lengkap di Bagian 6")


---
<a id="bagian-5"></a>
## Bagian 5 — Exploratory Data Analysis (EDA)

EDA bertujuan memahami distribusi, pola, dan hubungan antar variabel sebelum membangun model.


In [ ]:
# ── 5a. Korelasi Matrix ───────────────────────────────────────────────────────
print("── 5a. KORELASI MATRIX ──")
num_df = df_clean.select_dtypes(include=np.number)
corr_matrix = num_df.corr()
print("\n  Korelasi terhadap Burnout (diurutkan):")
corr_burnout = corr_matrix["Burnout"].drop("Burnout").sort_values(key=abs, ascending=False)
for feat, val in corr_burnout.items():
    direction = "↑ positif" if val > 0 else "↓ negatif"
    strength  = "🔴 Kuat" if abs(val) >= 0.4 else "🟡 Sedang" if abs(val) >= 0.2 else "🟢 Lemah"
    print(f"  {feat:<22}: {val:+.4f}  {direction}  {strength}")


In [ ]:
# ── 5b. Deteksi Outlier ───────────────────────────────────────────────────────
print("── 5b. DETEKSI OUTLIER (Metode IQR) ──")
print("""
  Nilai dianggap outlier jika berada di luar:
  Batas Bawah = Q1 - 1.5 × IQR
  Batas Atas  = Q3 + 1.5 × IQR
""")

outlier_cols = ["Age", "WorkHoursPerWeek", "StressLevel",
                "SatisfactionLevel", "Experience", "RemoteRatio"]
for col in outlier_cols:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    status = "✅ Bersih" if n_out == 0 else f"⚠️  {n_out} outlier ({n_out/len(df_clean)*100:.1f}%)"
    print(f"  {col:<22}: [{lo:>6.1f}, {hi:>6.1f}]  {status}")


In [ ]:
# ── 5c. Statistik Perbandingan Burnout vs Tidak Burnout ──────────────────────
print("── 5c. PERBANDINGAN STATISTIK: BURNOUT vs TIDAK BURNOUT ──")
print("""
  Kita bandingkan rata-rata fitur numerik antara dua kelompok
  untuk mendapatkan insight awal tentang pola yang membedakan mereka.
""")

group_stats = df_clean.groupby("Burnout")[
    ["WorkHoursPerWeek", "StressLevel", "SatisfactionLevel", "Age"]
].mean().round(2)

# Labeling
group_stats.index = ["Tidak Burnout (0)", "Burnout (1)"]
print(group_stats.to_string())

print("\n  Interpretasi:")
for col in ["WorkHoursPerWeek", "StressLevel", "SatisfactionLevel"]:
    m0 = df_clean[df_clean["Burnout"]==0][col].mean()
    m1 = df_clean[df_clean["Burnout"]==1][col].mean()
    diff = m1 - m0
    direction = "lebih tinggi" if diff > 0 else "lebih rendah"
    emoji = "🔺" if diff > 0 else "🔻"
    print(f"  {emoji} {col}: karyawan burnout rata-rata {abs(diff):.2f} poin {direction}")


In [ ]:
# ── 5d. Skewness & Kurtosis ───────────────────────────────────────────────────
print("── 5d. SKEWNESS & KURTOSIS ──")
print("""
  Skewness: asimetri distribusi (0 = simetris)
    > 0  = ekor ke kanan (positively skewed)
    < 0  = ekor ke kiri  (negatively skewed)
  Kurtosis: ketajaman puncak distribusi (0 = normal)
    > 0  = distribusi lebih lancip dari normal (leptokurtik)
    < 0  = distribusi lebih datar dari normal (platikurtik)
""")

skew_data = []
for col in ["Age", "WorkHoursPerWeek", "StressLevel", "SatisfactionLevel", "Experience"]:
    skew_data.append({
        "Fitur": col,
        "Skewness": round(df_clean[col].skew(), 4),
        "Kurtosis": round(df_clean[col].kurtosis(), 4),
        "Bentuk Distribusi": (
            "Simetris" if abs(df_clean[col].skew()) < 0.5 else
            "Skewed Kanan" if df_clean[col].skew() > 0 else "Skewed Kiri"
        )
    })

pd.DataFrame(skew_data).set_index("Fitur")


---
<a id="bagian-6"></a>
## Bagian 6 — Visualisasi & Explanatory Analysis

Setiap visualisasi dipilih berdasarkan tujuan analitisnya dan dijelaskan alasannya. 
Semua grafik juga disimpan sebagai file `.png` untuk kebutuhan laporan.


In [ ]:
# ── Fig 1: Distribusi Fitur Utama ────────────────────────────────────────────
# PILIHAN VISUALISASI: Histogram + KDE
# Alasan: Histogram menampilkan frekuensi nilai, KDE memperhalus kurva distribusi.
# Dengan pewarnaan berdasarkan Burnout, langsung terlihat perbedaan distribusi
# antara karyawan yang burnout vs tidak — mudah dipahami orang awam sekalipun.

features_plot = ["Age", "WorkHoursPerWeek", "StressLevel",
                 "SatisfactionLevel", "Experience", "RemoteRatio"]

fig1, axes = plt.subplots(2, 3, figsize=(18, 10))
fig1.suptitle(
    "Distribusi Fitur Utama Dataset Burnout Karyawan\n"
    "(Biru = Tidak Burnout  |  Merah = Burnout)",
    fontsize=15, fontweight="bold", y=1.01
)

for ax, col in zip(axes.flat, features_plot):
    sns.histplot(data=df_clean, x=col, hue="Burnout", bins=20, kde=True, ax=ax,
                 palette={0: COLOR_NO, 1: COLOR_YES})
    ax.set_title(f"Distribusi {col}", fontsize=12, fontweight="bold")
    ax.set_xlabel(col, fontsize=10)
    leg = ax.get_legend()
    if leg:
        leg.set_title("Status")
        for t, l in zip(leg.texts, ["Tidak Burnout", "Burnout"]):
            t.set_text(l)

plt.tight_layout()
fig1.savefig(f"{OUTPUT_DIR}/fig1_distribusi_fitur.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 1 tersimpan: fig1_distribusi_fitur.png")
print("""
Hasil & Insight Fig 1:
  • WorkHoursPerWeek: Karyawan burnout (merah) terkonsentrasi di jam kerja tinggi (>55 jam).
    Karyawan tidak burnout tersebar lebih merata di 30–60 jam.
  • StressLevel: Karyawan burnout sangat terkonsentrasi di stres level 8–10,
    sedangkan tidak burnout tersebar di seluruh range stres.
  • SatisfactionLevel: Karyawan burnout dominan di kepuasan 1–2 (sangat rendah).
    Konfirmasi hipotesis: kepuasan rendah = risiko burnout tinggi.
  • Age & Experience: Distribusi relatif serupa antara dua kelompok,
    mengindikasikan usia & pengalaman bukan faktor dominan.
  • RemoteRatio: Distribusi hampir identik, efek remote work tidak kuat secara mandiri.
""")


In [ ]:
# ── Fig 2: Heatmap Korelasi ───────────────────────────────────────────────────
# PILIHAN VISUALISASI: Heatmap dengan Annotasi
# Alasan: Heatmap memungkinkan melihat korelasi SEMUA pasang fitur sekaligus
# dalam satu tampilan berwarna. Sangat efisien untuk mengidentifikasi
# multikolinearitas dan fitur yang berkorelasi kuat dengan target.

fig2, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, ax=ax,
    linewidths=0.5, annot_kws={"size": 10}
)
ax.set_title("Heatmap Korelasi Antar Fitur Numerik", fontsize=14, fontweight="bold")
plt.tight_layout()
fig2.savefig(f"{OUTPUT_DIR}/fig2_heatmap_korelasi.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 2 tersimpan: fig2_heatmap_korelasi.png")
print("""
Hasil & Insight Fig 2:
  • StressLevel berkorelasi POSITIF terkuat dengan Burnout (+0.32)
    → Makin stres, makin tinggi risiko burnout.
  • SatisfactionLevel berkorelasi NEGATIF terkuat dengan Burnout (-0.23)
    → Makin puas, makin rendah risiko burnout.
  • WorkHoursPerWeek berkorelasi POSITIF dengan Burnout (+0.23)
    → Jam kerja lebih banyak → risiko burnout lebih tinggi.
  • Age dan Experience berkorelasi tinggi satu sama lain (+0.64)
    → Multikolinearitas! Kedua fitur membawa informasi serupa.
  • RemoteRatio hampir tidak berkorelasi dengan Burnout (≈0)
    → Remote work sendirian tidak cukup menjelaskan burnout.
""")


In [ ]:
# ── Fig 3: Burnout Rate per Kategori (BQ2 & BQ3) ────────────────────────────
# PILIHAN VISUALISASI: Bar Chart (Horizontal & Vertikal)
# Alasan: Bar chart ideal untuk membandingkan nilai diskret antar kategori.
# Horizontal untuk JobRole (label panjang lebih terbaca), vertikal untuk Gender.

fig3, axes = plt.subplots(1, 2, figsize=(16, 6))
fig3.suptitle("Burnout Rate per Kategori (Menjawab BQ2 & BQ3)",
              fontsize=14, fontweight="bold")

# BQ2 — per JobRole
br_role_s = br_role.sort_values()
colors3a = [PALETTE[i % len(PALETTE)] for i in range(len(br_role_s))]
bars3a = axes[0].barh(br_role_s.index, br_role_s.values, color=colors3a)
axes[0].set_xlabel("Burnout Rate (%)", fontsize=11)
axes[0].set_title("BQ2: Burnout Rate per JobRole", fontsize=12, fontweight="bold")
for bar, val in zip(bars3a, br_role_s.values):
    axes[0].text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")

# BQ3 — per Gender
bars3b = axes[1].bar(br_gender.index, br_gender.values,
                     color=[COLOR_NO, COLOR_YES][:len(br_gender)],
                     edgecolor="black", width=0.5)
axes[1].set_ylabel("Burnout Rate (%)", fontsize=11)
axes[1].set_title("BQ3: Burnout Rate per Gender", fontsize=12, fontweight="bold")
for bar, val in zip(bars3b, br_gender.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                 f"{val:.1f}%", ha="center", fontsize=13, fontweight="bold")

plt.tight_layout()
fig3.savefig(f"{OUTPUT_DIR}/fig3_burnout_kategori.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 3 tersimpan: fig3_burnout_kategori.png")
print(f"""
Hasil & Insight Fig 3:
  BQ2 — per JobRole:
  • Sales memiliki burnout rate tertinggi ({br_role.index[0]}: {br_role.values[0]:.1f}%).
  • Perbedaan antar role tidak terlalu besar, menandakan burnout adalah
    isu lintas jabatan yang perlu ditangani secara menyeluruh.
  • HR perlu memantau semua departemen, bukan hanya satu divisi tertentu.

  BQ3 — per Gender:
  • Perbedaan burnout rate antar gender sangat kecil ({abs(br_gender.max()-br_gender.min()):.2f}%).
  • Gender BUKAN faktor diskriminatif yang signifikan.
  • Intervensi HR tidak perlu membedakan berdasarkan gender.
""")


In [ ]:
# ── Fig 4: Boxplot Perbandingan (BQ4 & BQ7) ─────────────────────────────────
# PILIHAN VISUALISASI: Boxplot
# Alasan: Boxplot dalam satu grafik menampilkan median, Q1, Q3, dan outlier.
# Sangat efektif untuk membandingkan distribusi dua kelompok secara visual.
# Orang awam pun dapat langsung melihat perbedaan "rata-rata" antar grup.

df_box = df_clean.copy()
df_box["Status"] = df_box["Burnout"].map({0: "Tidak Burnout", 1: "Burnout"})

fig4, axes = plt.subplots(1, 3, figsize=(18, 6))
fig4.suptitle("Perbandingan Distribusi Fitur: Burnout vs Tidak Burnout (BQ4 & BQ7)",
              fontsize=14, fontweight="bold")

for ax, (col, title) in zip(axes, [
    ("WorkHoursPerWeek",  "Jam Kerja per Minggu"),
    ("StressLevel",        "Tingkat Stres (1–10)"),
    ("SatisfactionLevel",  "Kepuasan Kerja (1–5)"),
]):
    sns.boxplot(data=df_box, x="Status", y=col,
                palette={"Tidak Burnout": COLOR_NO, "Burnout": COLOR_YES}, ax=ax)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    # Annotation median
    for i, grp in enumerate(["Tidak Burnout", "Burnout"]):
        med = df_box[df_box["Status"] == grp][col].median()
        ax.text(i, med, f" Med={med:.1f}", va="center", fontsize=8.5,
                color="white", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="gray", alpha=0.7))

plt.tight_layout()
fig4.savefig(f"{OUTPUT_DIR}/fig4_boxplot_burnout.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 4 tersimpan: fig4_boxplot_burnout.png")
print("""
Hasil & Insight Fig 4:
  • WorkHoursPerWeek: Karyawan burnout (merah) memiliki median jam kerja
    jauh lebih tinggi (±60 jam) vs tidak burnout (±48 jam).
    Perbedaan nyata dan signifikan!
  • StressLevel: Karyawan burnout memiliki median stres ±9 vs ±5.
    Distribusi sangat sempit pada karyawan burnout → mereka hampir semuanya
    berada di stres sangat tinggi.
  • SatisfactionLevel: Karyawan burnout memiliki median kepuasan ±2 vs ±3.
    Konfirmasi BQ7: kepuasan rendah berkorelasi kuat dengan burnout.
""")


In [ ]:
# ── Fig 5: Distribusi Risk Level ─────────────────────────────────────────────
# PILIHAN VISUALISASI: Pie Chart + Bar Chart kombinasi
# Alasan: Pie chart menunjukkan komposisi proporsi secara intuitif.
# Bar chart melengkapi dengan nilai absolut. Kombinasi keduanya
# memberikan gambaran lengkap yang mudah dipahami semua kalangan.

risk_counts = df_clean["RiskLevel"].value_counts()
order_risk  = ["Rendah", "Sedang", "Tinggi"]
colors_risk = ["#4CAF50", "#FF9800", "#F44336"]
vals_risk   = [risk_counts.get(r, 0) for r in order_risk]

fig5, axes = plt.subplots(1, 2, figsize=(14, 6))
fig5.suptitle("Distribusi Tingkat Risiko Burnout Karyawan (BQ1 & BQ5)",
              fontsize=14, fontweight="bold")

wedges, texts, autotexts = axes[0].pie(
    vals_risk, labels=order_risk, colors=colors_risk,
    autopct="%1.1f%%", startangle=140, pctdistance=0.85,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12}
)
for at in autotexts:
    at.set_fontweight("bold")
axes[0].set_title("Proporsi Risk Level", fontsize=12, fontweight="bold")

bars5 = axes[1].bar(order_risk, vals_risk, color=colors_risk, edgecolor="black")
axes[1].set_ylabel("Jumlah Karyawan", fontsize=11)
axes[1].set_title("Jumlah Karyawan per Risk Level", fontsize=12, fontweight="bold")
for bar, (r, val) in zip(bars5, zip(order_risk, vals_risk)):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 10,
                 f"{val:,}\n({val/len(df_clean)*100:.1f}%)",
                 ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
fig5.savefig(f"{OUTPUT_DIR}/fig5_risk_level.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 5 tersimpan: fig5_risk_level.png")
print(f"""
Hasil & Insight Fig 5:
  • Sebagian besar karyawan ({vals_risk[1]/len(df_clean)*100:.1f}%) berada di Risk Level SEDANG.
  • {vals_risk[2]/len(df_clean)*100:.1f}% karyawan sudah berada di Risk Level TINGGI — perlu perhatian segera!
  • Hanya {vals_risk[0]/len(df_clean)*100:.1f}% yang berada di zona aman (Rendah).
  • Ini berarti lebih dari 84% karyawan berpotensi membutuhkan intervensi HR.
""")


In [ ]:
# ── Fig 6: Scatter Plot (BQ5) ────────────────────────────────────────────────
# PILIHAN VISUALISASI: Scatter Plot + Hexbin
# Alasan: Scatter plot memperlihatkan hubungan 2 variabel kontinyu sekaligus,
# dengan pewarnaan grup ketiga (Burnout). Kita bisa melihat apakah titik-titik
# merah (burnout) mengelompok di area tertentu (jam tinggi + stres tinggi).

fig6, axes = plt.subplots(1, 2, figsize=(16, 6))
fig6.suptitle("Hubungan Jam Kerja, Stres & Burnout (Menjawab BQ5)",
              fontsize=14, fontweight="bold")

# Panel kiri: Scatter WorkHours vs Stress
colors_sc = df_clean["Burnout"].map({0: COLOR_NO, 1: COLOR_YES})
axes[0].scatter(df_clean["WorkHoursPerWeek"], df_clean["StressLevel"],
                c=colors_sc, alpha=0.45, s=25, edgecolors="none")
axes[0].axvline(50, color="gray", linestyle="--", linewidth=1.5, alpha=0.7, label="Batas 50 jam")
axes[0].axhline(7,  color="purple", linestyle=":",  linewidth=1.5, alpha=0.7, label="Stres Tinggi (≥7)")
axes[0].set_xlabel("Jam Kerja per Minggu", fontsize=11)
axes[0].set_ylabel("Tingkat Stres", fontsize=11)
axes[0].set_title("WorkHoursPerWeek vs StressLevel", fontsize=12, fontweight="bold")
legend_elem = [
    mpatches.Patch(color=COLOR_NO,  label="Tidak Burnout"),
    mpatches.Patch(color=COLOR_YES, label="Burnout"),
]
axes[0].legend(handles=legend_elem + [
    plt.Line2D([0],[0], color="gray", linestyle="--", label="Batas 50 jam"),
    plt.Line2D([0],[0], color="purple", linestyle=":", label="Stres ≥7"),
], fontsize=9)

# Panel kanan: Hexbin density map
hb = axes[1].hexbin(df_clean["WorkHoursPerWeek"], df_clean["StressLevel"],
                    C=df_clean["Burnout"], gridsize=15,
                    cmap="RdYlGn_r", reduce_C_function=np.mean)
fig6.colorbar(hb, ax=axes[1], label="Rata-rata Burnout Rate")
axes[1].set_xlabel("Jam Kerja per Minggu", fontsize=11)
axes[1].set_ylabel("Tingkat Stres", fontsize=11)
axes[1].set_title("Density Map: Burnout Rate\n(Merah gelap = burnout rate tinggi)",
                  fontsize=12, fontweight="bold")

plt.tight_layout()
fig6.savefig(f"{OUTPUT_DIR}/fig6_scatter_burnout.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 6 tersimpan: fig6_scatter_burnout.png")
print("""
Hasil & Insight Fig 6:
  • Titik MERAH (burnout) sangat terkonsentrasi di kuadran kanan-atas:
    jam kerja > 50 DAN stres level > 7.
  • Density map (hexbin) mengkonfirmasi: area jam kerja tinggi + stres tinggi
    memiliki burnout rate paling tinggi (warna merah gelap).
  • Kombinasi dua faktor ini jauh lebih prediktif dibandingkan masing-masing faktor secara sendiri.
    Karyawan jam kerja tinggi TAPI stres rendah = masih aman.
  • Insight untuk HR: monitoring harus fokus pada karyawan yang SEKALIGUS
    memiliki jam kerja tinggi dan stres tinggi.
""")


In [ ]:
# ── Fig 7: Explanatory Analysis Komprehensif ─────────────────────────────────
# PILIHAN VISUALISASI: Multi-panel summary
# Alasan: Satu figure besar yang merangkum semua temuan utama sekaligus.
# Ideal untuk presentasi ke stakeholder non-teknis. Mudah dibawa ke rapat.

fig7, axes = plt.subplots(2, 3, figsize=(20, 12))
fig7.suptitle(
    "Ringkasan Explanatory Analysis — 7 Business Questions\n"
    "Burnout Risk Prediction System | CC26-PSU335",
    fontsize=15, fontweight="bold", y=1.01
)

# A: BQ1 — Proporsi Burnout
ax = axes[0, 0]
counts_bq1 = [int(df_clean["Burnout"].sum()), int((df_clean["Burnout"]==0).sum())]
labels_bq1 = [f"Burnout\n({br_overall:.1f}%)", f"Tidak Burnout\n({100-br_overall:.1f}%)"]
ax.pie(counts_bq1, labels=labels_bq1, colors=[COLOR_YES, COLOR_NO],
       startangle=90, autopct="%1.1f%%", pctdistance=0.7,
       wedgeprops={"edgecolor": "white", "linewidth": 2})
ax.set_title("BQ1: Proporsi Karyawan Burnout", fontsize=11, fontweight="bold")

# B: BQ2 — per JobRole
ax = axes[0, 1]
br_role_sorted = br_role.sort_values(ascending=True)
colors_b = sns.color_palette("husl", len(br_role_sorted))
ax.barh(br_role_sorted.index, br_role_sorted.values, color=colors_b)
ax.set_xlabel("Burnout Rate (%)", fontsize=10)
ax.set_title("BQ2: Burnout Rate per JobRole", fontsize=11, fontweight="bold")
for i, (r, v) in enumerate(br_role_sorted.items()):
    ax.text(v + 0.05, i, f"{v:.1f}%", va="center", fontsize=9)

# C: BQ3 — per Gender
ax = axes[0, 2]
bars_g = ax.bar(br_gender.index, br_gender.values,
                color=[COLOR_NO, COLOR_YES][:len(br_gender)], edgecolor="black", width=0.5)
ax.set_ylabel("Burnout Rate (%)", fontsize=10)
ax.set_title("BQ3: Burnout Rate per Gender", fontsize=11, fontweight="bold")
for bar, val in zip(bars_g, br_gender.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

# D: BQ4 — WorkHours vs Burnout
ax = axes[1, 0]
bins_wh = [0, 40, 50, 100]
labels_wh = ["<40 jam", "40-50 jam", ">50 jam"]
df_clean["WorkGroup"] = pd.cut(df_clean["WorkHoursPerWeek"], bins=bins_wh, labels=labels_wh)
br_work = df_clean.groupby("WorkGroup", observed=True)["Burnout"].mean() * 100
bars_d = ax.bar(br_work.index, br_work.values,
                color=[PALETTE[0], PALETTE[3], PALETTE[1]], edgecolor="black")
ax.set_ylabel("Burnout Rate (%)", fontsize=10)
ax.set_title("BQ4: Burnout per Kelompok Jam Kerja", fontsize=11, fontweight="bold")
for bar, val in zip(bars_d, br_work.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
            f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
df_clean = df_clean.drop(columns=["WorkGroup"])

# E: BQ6 — Korelasi vs Burnout
ax = axes[1, 1]
corr_sorted = corr_burnout.sort_values()
colors_e = [COLOR_YES if v > 0 else COLOR_NO for v in corr_sorted.values]
ax.barh(corr_sorted.index, corr_sorted.values, color=colors_e)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Pearson Correlation", fontsize=10)
ax.set_title("BQ6: Korelasi Fitur terhadap Burnout", fontsize=11, fontweight="bold")

# F: BQ7 — Kepuasan vs Stres (hexbin)
ax = axes[1, 2]
hb7 = ax.hexbin(df_clean["SatisfactionLevel"], df_clean["StressLevel"],
                C=df_clean["Burnout"], gridsize=15,
                cmap="RdYlGn_r", reduce_C_function=np.mean)
fig7.colorbar(hb7, ax=ax, label="Rata-rata Burnout Rate")
ax.set_xlabel("Kepuasan Kerja", fontsize=10)
ax.set_ylabel("Tingkat Stres", fontsize=10)
ax.set_title("BQ7: Kepuasan vs Stres → Burnout Rate\n(Gelap = burnout tinggi)",
             fontsize=11, fontweight="bold")

plt.tight_layout()
fig7.savefig(f"{OUTPUT_DIR}/fig7_explanatory.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 7 tersimpan: fig7_explanatory.png")
print("\n🎯 Semua 7 Business Questions telah divisualisasikan!")


---
<a id="bagian-7"></a>
## Bagian 7 — Feature Engineering & Data Dictionary

Feature Engineering menciptakan fitur baru yang lebih informatif dari fitur yang sudah ada, 
menangkap interaksi antar variabel dan domain knowledge.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# BAGIAN 7: FEATURE ENGINEERING & DATA DICTIONARY
# ═══════════════════════════════════════════════════════════════

df_model = df_clean.copy()

# ── Encoding Kategorikal ───────────────────────────────────────────────────────
print("── ENCODING KATEGORIKAL ──")
le_gender = LabelEncoder()
le_role   = LabelEncoder()
df_model["Gender_enc"]  = le_gender.fit_transform(df_model["Gender"])
df_model["JobRole_enc"] = le_role.fit_transform(df_model["JobRole"])

print(f"  Gender  : {dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))}")
print(f"  JobRole : {dict(zip(le_role.classes_, le_role.transform(le_role.classes_)))}")
print("""
  Mengapa Label Encoding (bukan One-Hot)?
  → Model tree-based bekerja baik dengan label encoding.
    One-Hot akan menambah dimensi fitur tidak perlu untuk 2 & 5 kategori ini.
""")

# ── Fitur Rekayasa ─────────────────────────────────────────────────────────────
print("── FITUR REKAYASA ──")

# Fitur 1
df_model["StressWorkRatio"] = df_model["StressLevel"] / (df_model["WorkHoursPerWeek"] + 1)
print("  ✓ StressWorkRatio = StressLevel / (WorkHoursPerWeek + 1)")
print("    → Intensitas stres per unit jam kerja. Karyawan stres tinggi dalam")
print("      jam sedikit vs banyak memiliki karakteristik risiko yang berbeda.\n")

# Fitur 2
df_model["WorkLifeScore"] = df_model["SatisfactionLevel"] * (1 - df_model["RemoteRatio"] / 100)
print("  ✓ WorkLifeScore = SatisfactionLevel × (1 - RemoteRatio/100)")
print("    → Mengukur keseimbangan kepuasan kerja dengan tingkat kehadiran fisik.")
print("      Kepuasan rendah + sedikit fleksibilitas remote = lebih rentan burnout.\n")

# Fitur 3
df_model["HighRiskFlag"] = (
    (df_model["StressLevel"] >= 7) & (df_model["WorkHoursPerWeek"] >= 50)
).astype(int)
n_hr = df_model["HighRiskFlag"].sum()
print(f"  ✓ HighRiskFlag = 1 jika StressLevel≥7 DAN WorkHoursPerWeek≥50")
print(f"    → Flag karyawan 'double high-risk'. Ada {n_hr:,} karyawan ({n_hr/len(df_model)*100:.1f}%).\n")

# Fitur 4
df_model["SeniorEmployee"] = (df_model["Experience"] >= 10).astype(int)
n_sen = df_model["SeniorEmployee"].sum()
print(f"  ✓ SeniorEmployee = 1 jika Experience ≥ 10 tahun")
print(f"    → Karyawan senior mungkin punya coping mechanism berbeda.")
print(f"      Ada {n_sen:,} karyawan senior ({n_sen/len(df_model)*100:.1f}%).\n")

# Fitur 5
df_model["StressCategory"] = pd.cut(
    df_model["StressLevel"], bins=[0, 3, 6, 10], labels=[0, 1, 2]
).astype(int)
print("  ✓ StressCategory = 0 (Rendah: 1–3) | 1 (Sedang: 4–6) | 2 (Tinggi: 7–10)")
print("    → Menangkap efek non-linear stres dalam 3 kelas.\n")

# Fitur 6
df_model["SatisfactionInverse"] = 5.0 - df_model["SatisfactionLevel"]
print("  ✓ SatisfactionInverse = 5 - SatisfactionLevel")
print("    → Ketidakpuasan kerja (berkorelasi positif dengan burnout).\n")

# Pilih fitur final
FEATURES = [
    "Age", "Experience", "WorkHoursPerWeek", "RemoteRatio",
    "SatisfactionLevel", "StressLevel",
    "Gender_enc", "JobRole_enc",
    "StressWorkRatio", "WorkLifeScore",
    "HighRiskFlag", "SeniorEmployee",
    "StressCategory", "SatisfactionInverse"
]
TARGET = "Burnout"

print(f"  Total fitur model: {len(FEATURES)}")
print("  Fitur:", FEATURES)


In [ ]:
# ── Data Dictionary ───────────────────────────────────────────────────────────
print("── DATA DICTIONARY ──")

data_dict = pd.DataFrame([
    ("Age",                "int",              "Original", "Usia karyawan dalam tahun (22–60)"),
    ("Gender",             "str Male/Female",  "Original", "Jenis kelamin karyawan"),
    ("JobRole",            "str",              "Original", "Posisi jabatan: Analyst/Engineer/HR/Manager/Sales"),
    ("Experience",         "int",              "Original", "Lama pengalaman kerja dalam tahun (0–39)"),
    ("WorkHoursPerWeek",   "int",              "Original", "Rata-rata jam kerja per minggu"),
    ("RemoteRatio",        "int %",            "Original", "Persentase waktu kerja remote (0–100%)"),
    ("SatisfactionLevel",  "float 1.0–5.0",   "Original", "Skor kepuasan kerja (1=sangat tidak puas, 5=sangat puas)"),
    ("StressLevel",        "int 1–10",         "Original", "Tingkat stres kerja (1=sangat rendah, 10=sangat tinggi)"),
    ("Burnout",            "int 0/1",          "Target",   "Label target: 0=Tidak Burnout, 1=Burnout"),
    ("RiskLevel",          "str 3 kelas",      "Rekayasa", "Tingkat risiko: Rendah / Sedang / Tinggi"),
    ("Gender_enc",         "int 0/1",          "Rekayasa", "Encoding: Female=0, Male=1"),
    ("JobRole_enc",        "int 0–4",          "Rekayasa", "Encoding Label untuk JobRole"),
    ("StressWorkRatio",    "float",            "Rekayasa", "StressLevel / (WorkHoursPerWeek+1)"),
    ("WorkLifeScore",      "float",            "Rekayasa", "SatisfactionLevel × (1 – RemoteRatio/100)"),
    ("HighRiskFlag",       "int 0/1",          "Rekayasa", "1 jika StressLevel≥7 AND WorkHoursPerWeek≥50"),
    ("SeniorEmployee",     "int 0/1",          "Rekayasa", "1 jika Experience≥10 tahun"),
    ("StressCategory",     "int 0/1/2",        "Rekayasa", "Kategori stres: 0=Rendah, 1=Sedang, 2=Tinggi"),
    ("SatisfactionInverse","float",            "Rekayasa", "5 - SatisfactionLevel: mengukur ketidakpuasan"),
], columns=["Kolom", "Tipe", "Jenis", "Deskripsi"])

# Simpan ke CSV
data_dict.to_csv(f"{OUTPUT_DIR}/data_dictionary.csv", index=False)
print("  ✅ Data Dictionary tersimpan: data_dictionary.csv\n")

# Tampilkan dengan styling
data_dict.style.applymap(
    lambda v: "background-color: #d4edda" if v == "Rekayasa"
    else ("background-color: #cce5ff" if v == "Target" else ""),
    subset=["Jenis"]
)


---
<a id="bagian-8"></a>
## Bagian 8 — Persiapan Data untuk Model & Evaluasi

Tahap akhir mempersiapkan data siap model dan mengevaluasi 4 algoritma Machine Learning.


In [ ]:
# ── Scaling & Split Data ─────────────────────────────────────────────────────
print("── PERSIAPAN DATA MODEL ──")

X = df_model[FEATURES].copy()
y = df_model[TARGET].copy()

print(f"  Ukuran X: {X.shape} | Ukuran y: {y.shape}")
print(f"  Distribusi y: {dict(y.value_counts().items())}")

# Scaling
scaler      = StandardScaler()
X_scaled    = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)

print("\n  StandardScaler diterapkan: mean=0, std=1 untuk setiap fitur")
print("  → Penting untuk Logistic Regression & model berbasis jarak")
print("  → Scaling konsisten meski tree-based tidak sensitif")

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n  Train: {len(X_train):,} baris ({len(X_train)/len(X)*100:.0f}%) | "
      f"Burnout rate: {y_train.mean()*100:.1f}%")
print(f"  Test : {len(X_test):,} baris  ({len(X_test)/len(X)*100:.0f}%)  | "
      f"Burnout rate: {y_test.mean()*100:.1f}%")
print("  → Stratified split ✅ proporsi kelas identik di train & test")


In [ ]:
# ── Training & Evaluasi 4 Model ──────────────────────────────────────────────
print("── TRAINING & EVALUASI MODEL (5-Fold CV) ──\n")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Decision Tree"       : DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred   = model.predict(X_test)
    y_proba  = model.predict_proba(X_test)[:, 1]
    cv_f1    = cross_val_score(model, X_scaled_df, y, cv=cv, scoring="f1")

    results[name] = {
        "Accuracy":   accuracy_score(y_test, y_pred),
        "F1":         f1_score(y_test, y_pred, zero_division=0),
        "Precision":  precision_score(y_test, y_pred, zero_division=0),
        "Recall":     recall_score(y_test, y_pred, zero_division=0),
        "AUC-ROC":    roc_auc_score(y_test, y_proba),
        "CV_F1_Mean": cv_f1.mean(),
        "CV_F1_Std":  cv_f1.std(),
    }
    print(f"  ── {name} ──")
    for k, v in results[name].items():
        print(f"     {k:<12}: {v:.4f}")
    print()

results_df = pd.DataFrame(results).T.round(4)
print("─" * 60)
best_name  = results_df["AUC-ROC"].idxmax()
best_model = models[best_name]
print(f"  🏆 Model terbaik (AUC-ROC): {best_name}")


In [ ]:
# ── Rangkuman & Classification Report ────────────────────────────────────────
print("── RANGKUMAN PERFORMA MODEL ──")
display(results_df[["Accuracy","F1","Precision","Recall","AUC-ROC","CV_F1_Mean","CV_F1_Std"]]
        .style.highlight_max(axis=0, color="#d4f0d4")
        .format("{:.4f}"))

print(f"\n  Classification Report — {best_name}:")
y_pred_best = best_model.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=["Tidak Burnout", "Burnout"]))


In [ ]:
# ── Visualisasi Performa Model ────────────────────────────────────────────────
fig_model, axes = plt.subplots(1, 2, figsize=(16, 6))
fig_model.suptitle("Evaluasi Model ML — Perbandingan & Feature Importance",
                   fontsize=14, fontweight="bold")

# Panel kiri: Grouped bar per metrik
metrics_ev = ["Accuracy", "F1", "Precision", "Recall", "AUC-ROC"]
x     = np.arange(len(metrics_ev))
width = 0.18
for i, (name, row) in enumerate(results_df.iterrows()):
    vals = [float(row[m]) for m in metrics_ev]
    axes[0].bar(x + i*width, vals, width, label=name, color=PALETTE[i])
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(metrics_ev)
axes[0].set_ylim(0, 1.15)
axes[0].legend(fontsize=8)
axes[0].set_title("Perbandingan Metrik Evaluasi", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Score")

# Panel kanan: Feature Importance
if hasattr(best_model, "feature_importances_"):
    feat_imp = pd.DataFrame({
        "Fitur": FEATURES, "Importance": best_model.feature_importances_
    }).sort_values("Importance", ascending=True)
    colors_fi = ["#F44336" if v >= feat_imp["Importance"].quantile(0.75) else
                 "#FF9800" if v >= feat_imp["Importance"].quantile(0.5) else "#4CAF50"
                 for v in feat_imp["Importance"]]
    axes[1].barh(feat_imp["Fitur"], feat_imp["Importance"], color=colors_fi)
    axes[1].set_xlabel("Importance Score")
    axes[1].set_title(f"Feature Importance — {best_name}", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_model_eval.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Visualisasi model tersimpan")


---
<a id="bagian-9"></a>
## Bagian 9 — A/B Testing

A/B Testing memvalidasi secara statistik apakah perbedaan yang diamati (antar model atau antar kelompok karyawan) benar-benar bermakna atau hanya kebetulan.

**Metode:** Two-Sample Independent t-test | **α = 0.05**

| Test | Grup A | Grup B | Hipotesis |
|------|--------|--------|-----------|
| Test 1 | Logistic Regression | Random Forest | Apakah ada perbedaan F1-Score? |
| Test 2 | Random Forest | Gradient Boosting | Apakah ada perbedaan F1-Score? |
| Test 3 | Jam kerja < 40 | Jam kerja ≥ 50 | Apakah jam kerja tinggi → burnout lebih tinggi? |


In [ ]:
# ── A/B Testing — Hitung CV F1 Scores ───────────────────────────────────────
print("── MENGHITUNG CV F1 SCORES (5-fold) ──")
print("  (Proses ini membutuhkan ~30 detik, mohon tunggu...)\n")

cv_lr = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    X_scaled_df, y, cv=cv, scoring="f1"
)
cv_rf = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1),
    X_scaled_df, y, cv=cv, scoring="f1"
)
cv_gb = cross_val_score(
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    X_scaled_df, y, cv=cv, scoring="f1"
)

print("  CV F1 Scores per Model:")
print(f"    Logistic Regression : {cv_lr.round(4).tolist()}")
print(f"    Random Forest       : {cv_rf.round(4).tolist()}")
print(f"    Gradient Boosting   : {cv_gb.round(4).tolist()}")
print(f"\n  Mean ± Std:")
print(f"    LR : {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")
print(f"    RF : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")
print(f"    GB : {cv_gb.mean():.4f} ± {cv_gb.std():.4f}")


In [ ]:
# ── A/B Test 1: LR vs RF ─────────────────────────────────────────────────────
print("═" * 50)
print("  A/B TEST 1: Logistic Regression vs Random Forest")
print("═" * 50)
print("""
  H₀: Tidak ada perbedaan F1-Score antara LR dan RF
  H₁: RF memiliki F1-Score lebih tinggi (one-sided)
  Metode: Two-sample independent t-test
""")

t1, p1 = stats.ttest_ind(cv_lr, cv_rf)
print(f"  t-statistic : {t1:.4f}")
print(f"  p-value     : {p1:.6f}")
print(f"  α = 0.05")
print()
if p1 < 0.05:
    winner1 = "Random Forest" if cv_rf.mean() > cv_lr.mean() else "Logistic Regression"
    print(f"  🔴 TOLAK H₀ — Perbedaan SIGNIFIKAN (p < 0.05)")
    print(f"  🏆 Pemenang: {winner1} (mean F1 lebih tinggi)")
else:
    print(f"  🟢 GAGAL TOLAK H₀ — Tidak ada perbedaan signifikan (p ≥ 0.05)")


In [ ]:
# ── A/B Test 2: RF vs GB ─────────────────────────────────────────────────────
print("═" * 50)
print("  A/B TEST 2: Random Forest vs Gradient Boosting")
print("═" * 50)
print("""
  H₀: Tidak ada perbedaan F1-Score antara RF dan GB
  H₁: Salah satu model lebih unggul secara signifikan
""")

t2, p2 = stats.ttest_ind(cv_rf, cv_gb)
print(f"  t-statistic : {t2:.4f}")
print(f"  p-value     : {p2:.6f}")
print()
if p2 < 0.05:
    winner2 = "Gradient Boosting" if cv_gb.mean() > cv_rf.mean() else "Random Forest"
    print(f"  🔴 TOLAK H₀ — Perbedaan signifikan")
    print(f"  🏆 Pemenang: {winner2}")
else:
    print(f"  🟢 GAGAL TOLAK H₀ — Tidak ada perbedaan signifikan")
    print(f"  → Kedua model memiliki performa setara. Pilih berdasarkan interpretability.")


In [ ]:
# ── A/B Test 3: Jam Kerja Tinggi vs Rendah ──────────────────────────────────
print("═" * 50)
print("  A/B TEST 3: Burnout Rate — Jam Kerja Tinggi vs Rendah")
print("═" * 50)
print("""
  H₀: Tidak ada perbedaan burnout rate antara kelompok jam kerja tinggi & rendah
  H₁: Jam kerja tinggi (≥50 jam) → burnout rate lebih tinggi secara signifikan
""")

high_grp = df_clean[df_clean["WorkHoursPerWeek"] >= 50]["Burnout"].values
low_grp  = df_clean[df_clean["WorkHoursPerWeek"] <  40]["Burnout"].values

print(f"  Kelompok ≥50 jam: burnout rate {high_grp.mean()*100:.2f}%  (n={len(high_grp):,})")
print(f"  Kelompok <40 jam: burnout rate {low_grp.mean()*100:.2f}%  (n={len(low_grp):,})")

t3, p3 = stats.ttest_ind(high_grp, low_grp)
print(f"\n  t-statistic : {t3:.4f}")
print(f"  p-value     : {p3:.6f}")
print()
if p3 < 0.05:
    print(f"  🔴 TOLAK H₀ — Jam kerja tinggi TERBUKTI meningkatkan burnout rate secara signifikan!")
    print(f"  📌 Rekomendasi: Implementasikan kebijakan batas jam kerja maksimal 45–50 jam/minggu.")
else:
    print(f"  🟢 GAGAL TOLAK H₀ — Tidak ada bukti perbedaan signifikan")


In [ ]:
# ── Visualisasi A/B Testing ───────────────────────────────────────────────────
fig8, axes = plt.subplots(1, 3, figsize=(18, 6))
fig8.suptitle("A/B Testing — Validasi Statistik Perbandingan Model & Faktor Burnout",
              fontsize=14, fontweight="bold")

# Panel A: Boxplot CV F1
model_names_ab = ["Logistic\nRegression", "Random\nForest", "Gradient\nBoosting"]
bp = axes[0].boxplot([cv_lr, cv_rf, cv_gb], labels=model_names_ab, patch_artist=True,
                     medianprops={"linewidth": 2, "color": "black"})
for patch, color in zip(bp["boxes"], PALETTE[:3]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].set_ylabel("F1 Score (CV 5-fold)")
axes[0].set_title(f"Test 1 & 2: Distribusi CV F1\nLR vs RF: p={p1:.4f} | RF vs GB: p={p2:.4f}", fontsize=10)
axes[0].yaxis.grid(True, alpha=0.5)

# Panel B: Mean F1 + Error bar
means = [cv_lr.mean(), cv_rf.mean(), cv_gb.mean()]
stds  = [cv_lr.std(), cv_rf.std(), cv_gb.std()]
axes[1].bar(model_names_ab, means, yerr=stds, color=PALETTE[:3],
            capsize=7, edgecolor="black", error_kw={"linewidth": 2})
axes[1].set_ylabel("Mean F1 Score")
axes[1].set_title("Mean F1 ± Std Dev\n(Error bar = variasi antar fold)", fontsize=10)
axes[1].set_ylim(0, max(means) * 1.35)
for i, (m, s) in enumerate(zip(means, stds)):
    axes[1].text(i, m + s + 0.005, f"{m:.4f}", ha="center", fontsize=10, fontweight="bold")

# Panel C: Burnout rate per kelompok jam kerja
labels_c = [f"<40 jam\n(n={len(low_grp):,})", f"≥50 jam\n(n={len(high_grp):,})"]
rates_c  = [low_grp.mean()*100, high_grp.mean()*100]
bars_c   = axes[2].bar(labels_c, rates_c, color=["#2196F3","#F44336"], edgecolor="black")
axes[2].set_ylabel("Burnout Rate (%)")
sig_lbl  = "✅ Signifikan (p < 0.05)" if p3 < 0.05 else "❌ Tidak Signifikan"
axes[2].set_title(f"Test 3: Burnout Rate per Kelompok Jam Kerja\np={p3:.4f} | {sig_lbl}", fontsize=10)
for bar, val in zip(bars_c, rates_c):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.05,
                 f"{val:.2f}%", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
fig8.savefig(f"{OUTPUT_DIR}/fig8_ab_testing.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Fig 8 (A/B Testing) tersimpan: fig8_ab_testing.png")


---
<a id="bagian-10"></a>
## Bagian 10 — Simpan Artifacts

Menyimpan semua file output yang dibutuhkan oleh tim lain (AI Engineer untuk FastAPI, 
Full-Stack Developer untuk integrasi frontend, dan Streamlit dashboard).


In [ ]:
# ── Simpan Model Artifacts ───────────────────────────────────────────────────
model_artifacts = {
    "best_model"      : best_model,
    "best_model_name" : best_name,
    "scaler"          : scaler,
    "le_gender"       : le_gender,
    "le_role"         : le_role,
    "features"        : FEATURES,
    "results_df"      : results_df.to_dict(),
}

with open(f"{OUTPUT_DIR}/model_artifacts.pkl", "wb") as f:
    pickle.dump(model_artifacts, f)
print("  ✅ model_artifacts.pkl tersimpan")
print(f"     Isi: model ({best_name}), scaler, le_gender, le_role, features, results_df")

# Simpan df_clean
df_clean.to_csv(f"{OUTPUT_DIR}/df_clean.csv", index=False)
print("  ✅ df_clean.csv tersimpan")
print(f"     Shape: {df_clean.shape[0]:,} baris × {df_clean.shape[1]} kolom")


In [ ]:
# ── Ringkasan Final ───────────────────────────────────────────────────────────
print("=" * 70)
print("  ✅  PIPELINE DATA SCIENTIST — SELESAI SELURUHNYA")
print("=" * 70)

print(f"""
  📁 File Output yang Dihasilkan:
  ─────────────────────────────────────────────────────────
  📊 fig1_distribusi_fitur.png   → Distribusi semua fitur (Bagian 6)
  🔥 fig2_heatmap_korelasi.png  → Heatmap korelasi (Bagian 6)
  📊 fig3_burnout_kategori.png  → Burnout per JobRole & Gender (BQ2, BQ3)
  📦 fig4_boxplot_burnout.png   → Boxplot perbandingan (BQ4, BQ7)
  🎯 fig5_risk_level.png        → Distribusi Risk Level (BQ1, BQ5)
  🔵 fig6_scatter_burnout.png   → Scatter WorkHours vs Stress (BQ5)
  📋 fig7_explanatory.png       → Ringkasan 7 Business Questions
  🧪 fig8_ab_testing.png        → Validasi statistik A/B Testing
  📈 fig_model_eval.png         → Evaluasi & Feature Importance model
  📄 data_dictionary.csv        → Kamus data lengkap (Bagian 7)
  🤖 model_artifacts.pkl        → Model + scaler + encoder (siap FastAPI)
  📋 df_clean.csv               → Dataset bersih siap pakai

  🚀 Langkah Selanjutnya:
  ─────────────────────────────────────────────────────────
  1. Jalankan Dashboard Streamlit:
     → streamlit run dashboard_burnout.py

  2. Deploy ke Streamlit Cloud:
     a. Push semua file ke GitHub
     b. Buka: https://streamlit.io/cloud
     c. New App → pilih repository → main file: dashboard_burnout.py
     d. Deploy! (sekitar 2–5 menit)

  3. Integrasi AI Engineer (FastAPI):
     → model_artifacts.pkl siap untuk inference endpoint
     → Contoh load: artifacts = pickle.load(open('model_artifacts.pkl','rb'))
     →              model = artifacts['best_model']

  4. Requirements untuk deployment:
     streamlit, pandas, numpy, matplotlib, seaborn, scikit-learn, scipy
""")
